# Aesthetic AI — Kaggle Training Notebook

**Prerequisites:**
- Add your `aesthetic-pairs-queue` Kaggle Dataset as input (contains `queue.db`)
- Enable GPU accelerator: T4 x1
- Run cells **one at a time** and verify each passes before proceeding

In [ ]:
# Cell 1: Clone repo + install deps

GITHUB_REPO = "https://github.com/krutckwang/aesthetic-ai.git"
REPO_DIR = "/kaggle/working/aesthetic-ai"

import os, subprocess, sys, numpy as np

# Patch PIL._util in memory BEFORE importing torchvision.
# Kaggle's base image has a mixed Pillow state: _util.py is from 10.x (no
# is_directory/is_path) but ImageFont.py is from 9.x (still imports them).
import PIL._util as _pil_util
if not hasattr(_pil_util, 'is_directory'):
    import os as _os
    from pathlib import Path as _Path
    _pil_util.is_directory = lambda fp: _os.path.isdir(fp)
    _pil_util.is_path = lambda f: isinstance(f, (bytes, str, _Path))

import torch, torchvision
from PIL import Image as _PIL

# Pin torch, torchvision, Pillow, numpy so the main pip install cannot downgrade them.
with open('/tmp/kaggle_constraints.txt', 'w') as f:
    f.write(f"torch=={torch.__version__}\n")
    f.write(f"torchvision=={torchvision.__version__}\n")
    f.write(f"Pillow=={_PIL.__version__}\n")
    f.write(f"numpy=={np.__version__}\n")

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

# Use sys.executable -m pip to guarantee installs land in the active interpreter's
# site-packages (plain 'pip' may resolve to a different location on Kaggle).
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', '/tmp/kaggle_constraints.txt',
    '--upgrade-strategy', 'only-if-needed',
    'diffusers>=0.27.0', 'peft>=0.11.0', 'accelerate>=0.30.0',
    'insightface>=0.7.3', 'onnxruntime>=1.18.0', 'mediapipe>=0.10.14',
    'sqlalchemy>=2.0.30', 'alembic>=1.13.1',
    'loguru', 'tqdm', 'pyyaml', 'python-dotenv',
], check=True)

print(f"torch:       {torch.__version__}   (must contain +cu128)")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {_PIL.__version__}")
print(f"numpy:       {np.__version__}")

In [ ]:
# Cell 2: Verify environment
# Expected: CUDA=True, VRAM >= 15 GB. Raise early if GPU not attached.

import torch, torchvision
from PIL import Image

print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {Image.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:         {props.name}")
    print(f"VRAM:        {props.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — enable T4 accelerator in Notebook settings")

In [ ]:
# Cell 3: Download images from queue.db

import sqlite3, httpx, json, os, sys, shutil, time, random
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/kaggle/working/aesthetic-ai')

QUEUE_DB_SRC = "/kaggle/input/datasets/yuyuanwang1/aesthetic-pairs-queue/queue.db"
shutil.copy2(QUEUE_DB_SRC, '/kaggle/working/queue.db')

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_name, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()
conn.close()

IMAGE_DIR = Path('/kaggle/working/aesthetic-ai/data/raw')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

existing = sum(
    1 for r in rows
    if (IMAGE_DIR / f"{r[0]}_before.jpg").exists()
    and (IMAGE_DIR / f"{r[0]}_after.jpg").exists()
)
print(f"Found {len(rows)} pairs in queue — {existing} already downloaded, skipping those")

downloaded, skipped = 0, 0
with httpx.Client(
    timeout=30, follow_redirects=True, verify=False,
    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
) as client:
    for row_id, before_url, after_url, source_name, metadata_str in tqdm(rows):
        b_path = IMAGE_DIR / f"{row_id}_before.jpg"
        a_path = IMAGE_DIR / f"{row_id}_after.jpg"
        try:
            if not b_path.exists():
                resp = client.get(before_url); resp.raise_for_status()
                b_path.write_bytes(resp.content)
            if not a_path.exists():
                resp = client.get(after_url); resp.raise_for_status()
                a_path.write_bytes(resp.content)
            downloaded += 1
            time.sleep(random.uniform(0.3, 0.8))
        except Exception:
            skipped += 1

print(f"Downloaded: {downloaded}  Failed/skipped: {skipped}")

In [ ]:
# Cell 4: Build manifest.json with face-detection filter + treatment label mapping

import subprocess, sys, sqlite3, json, torch
from pathlib import Path
from PIL import Image as PILImage
from tqdm import tqdm

# Install facenet-pytorch into the active interpreter.
# --no-deps skips its Pillow>=10.2,<10.3 constraint (we have 11.3, compatible at runtime).
# sys.executable -m pip ensures we hit the same site-packages this kernel uses.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'facenet-pytorch'],
    check=True
)

from facenet_pytorch import MTCNN

IMAGE_DIR    = Path('/kaggle/working/aesthetic-ai/data/raw')
MANIFEST_PATH = Path('/kaggle/working/aesthetic-ai/data/manifest.json')
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

SLUG_MAP = {
    'botulinum-toxin':                'botox',
    'botox':                          'botox',
    'lip-augmentation':               'lip_filler',
    'lip-augmentation---enhancement': 'lip_filler',
    'lip-enhancement':                'lip_filler',
    'facelift':                       'facelift',
    'face-lift':                      'facelift',
    'mini-facelift':                  'facelift',
    'brow-lift':                      'facelift',
    'neck-lift':                      'facelift',
    'eyelid-surgery':                 'blepharoplasty',
    'blepharoplasty':                 'blepharoplasty',
    'rhinoplasty':                    'rhinoplasty',
    'nose-surgery':                   'rhinoplasty',
    'dermal-fillers':                 'dermal_filler',
    'dermal-filler':                  'dermal_filler',
    'chin-augmentation':              'jawline_filler',
    'cheek-augmentation':             'dermal_filler',
    'under-eye-filler':               'under_eye_filler',
    'tear-trough':                    'under_eye_filler',
    'kybella':                        'kybella',
    'thread-lift':                    'thread_lift',
    'laser-skin-resurfacing':         'laser_resurfacing',
    'chemical-peel':                  'chemical_peel',
    'chemical-peels':                 'chemical_peel',
    'microneedling':                  'microneedling',
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn  = MTCNN(keep_all=False, device=device, min_face_size=40)

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_url, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()
conn.close()

def slug_from_url(url):
    return url.rstrip('/').split('/')[-1].split('?')[0] if url else ''

def extract_treatment(source_url, metadata_str):
    # 1. Prefer treatment stored in metadata by the crawler
    try:
        meta = json.loads(metadata_str or '{}')
        if meta.get('treatment_category'):
            return meta['treatment_category']
    except Exception:
        pass
    # 2. Search all URL path segments (not just the last)
    if source_url:
        for part in source_url.rstrip('/').split('/'):
            part = part.split('?')[0]
            if part in SLUG_MAP:
                return SLUG_MAP[part]
    return None

def has_face(path):
    try:
        boxes, _ = mtcnn.detect(PILImage.open(path).convert('RGB'))
        return boxes is not None and len(boxes) > 0
    except Exception:
        return False

records, no_image, no_face = [], 0, 0

for row_id, before_url, after_url, source_url, metadata_str in tqdm(rows, desc="face-detect"):
    b_path = IMAGE_DIR / f"{row_id}_before.jpg"
    a_path = IMAGE_DIR / f"{row_id}_after.jpg"
    if not (b_path.exists() and a_path.exists()):
        no_image += 1
        continue
    if not (has_face(b_path) and has_face(a_path)):
        no_face += 1
        continue

    treatment = extract_treatment(source_url, metadata_str)

    records.append({
        "pair_id":            row_id,
        "before_path":        str(b_path),
        "after_path":         str(a_path),
        "treatment_category": treatment,
        "treatment_brand":    None,
        "zone_codes":         [],
    })

MANIFEST_PATH.write_text(json.dumps(records, indent=2), encoding="utf-8")

labeled = sum(1 for r in records if r["treatment_category"])
print(f"Checked:           {len(rows)}")
print(f"Missing images:    {no_image}")
print(f"No face (skipped): {no_face}")
print(f"Manifest pairs:    {len(records)}")
print(f"Label coverage:    {labeled}/{len(records)} ({100*labeled/max(len(records),1):.1f}%)")

if len(records) < 200:
    raise RuntimeError(f"Only {len(records)} face pairs — too few to train.")

In [ ]:
# Cell 5: Train InstructPix2Pix + LoRA
# Run as a module (-m) so the project root is on sys.path and
# `from model.training.dataset import ...` resolves correctly.

import os
os.chdir('/kaggle/working/aesthetic-ai')

!python -m model.training.train \
    --manifest        data/manifest.json \
    --base_model      timbrooks/instruct-pix2pix \
    --output_dir      /kaggle/working/lora_output \
    --num_steps       15000 \
    --batch_size      2 \
    --mixed_precision fp16 \
    --lora_rank       16 \
    --save_every      500 \
    --num_workers     0